In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
real_df=pd.read_csv("/kaggle/input/datasets/toshangupta/adult-dataset/adult.csv")
real_df.replace('?', np.nan, inplace=True)
real_df.dropna(inplace=True)

syn_df=pd.read_csv("/kaggle/input/datasets/toshangupta/synthetic-adult-data-findiff/synthetic.csv")
#syn_df=syn_df.drop(columns=["Unnamed: 0"])
syn_df = syn_df[real_df.columns]
# Split real data
real_train, real_test = train_test_split(
    real_df,
    test_size=0.2,
    random_state=42
)

In [12]:
#numeric_cols=['age', 'fnlwgt', 'educational-num', 'capital-gain', 'capital-loss','hours-per-week']
categorical_cols=['workclass', 'education', 'marital-status', 'occupation','relationship', 'race', 'gender', 'native-country', 'income']

In [13]:
# import numpy as np
# import pandas as pd

# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import OrdinalEncoder
# from sklearn.metrics import accuracy_score, r2_score
# from sklearn.metrics import mean_squared_error
# from lightgbm import LGBMClassifier, LGBMRegressor


# def compute_structural_fidelity_tabstruct(
#     real_df,
#     synthetic_df,
#     test_size=0.2,
#     random_state=42,
# ):
#     feature_scores = {}
#     for target in real_df.columns:
#         print(target)
#         X_train_syn = synthetic_df.drop(columns=[target])
#         y_train_syn = synthetic_df[target]

#         X_test_real = real_df.drop(columns=[target])
#         y_test_real = real_df[target]
#         if target in categorical_cols:
#             # Classification
#             model = LGBMClassifier(
#                 n_estimators=200,
#                 learning_rate=0.05,
#                 max_depth=-1,
#                 random_state=random_state,
#             )

#             model.fit(X_train_syn, y_train_syn)
#             y_pred = model.predict(X_test_real)
#             score = accuracy_score(y_test_real, y_pred)

#         else:
#             # Regression
#             model = LGBMRegressor(
#                 n_estimators=200,
#                 learning_rate=0.05,
#                 max_depth=-1,
#                 random_state=random_state,
#             )

#             model.fit(X_train_syn, y_train_syn)
#             y_pred = model.predict(X_test_real)
#             score = np.sqrt(mean_squared_error(y_test_real, y_pred))
#         feature_scores[target] = score
#     return feature_scores
from sklearn.preprocessing import OrdinalEncoder
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.metrics import accuracy_score, mean_squared_error
import numpy as np


def compute_structural_fidelity_tabstruct(
    real_df,
    synthetic_df,
    test_size=0.2,
    random_state=42,
):

    feature_scores = {}

    for target in real_df.columns:
        print("Evaluating target:", target)

        X_train_syn = synthetic_df.drop(columns=[target]).copy()
        y_train_syn = synthetic_df[target]

        X_test_real = real_df.drop(columns=[target]).copy()
        y_test_real = real_df[target]

        # Identify categorical columns excluding current target
        cat_cols_current = [c for c in categorical_cols if c != target]

        # Fit encoder ONLY on synthetic training
        if len(cat_cols_current) > 0:
            encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

            encoder.fit(X_train_syn[cat_cols_current])

            X_train_syn[cat_cols_current] = encoder.transform(X_train_syn[cat_cols_current])
            X_test_real[cat_cols_current] = encoder.transform(X_test_real[cat_cols_current])

        # Convert everything to numeric
        X_train_syn = X_train_syn.astype(float)
        X_test_real = X_test_real.astype(float)

        # ---------- Classification ----------
        if target in categorical_cols:
            model = LGBMClassifier(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=-1,
                random_state=random_state,
                verbose=-1
            )

            model.fit(X_train_syn, y_train_syn)
            y_pred = model.predict(X_test_real)
            score = accuracy_score(y_test_real, y_pred)

        # ---------- Regression ----------
        else:
            model = LGBMRegressor(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=-1,
                random_state=random_state,
                verbose=-1
            )

            model.fit(X_train_syn, y_train_syn)
            y_pred = model.predict(X_test_real)
            score = np.sqrt(mean_squared_error(y_test_real, y_pred))

        feature_scores[target] = score

    return feature_scores


In [14]:
def compute_utilities(per_feature, per_feature_real, target_variable):
    per_feature_utility = {}
    for feature in per_feature:
        synth_perf = per_feature[feature]
        real_perf = per_feature_real[feature]

        if feature in categorical_cols:
            utility = synth_perf / real_perf
        else:
            utility = real_perf / synth_perf
        per_feature_utility[feature] = utility
    global_utility = sum(per_feature_utility.values()) / len(per_feature_utility)
    local_utility = per_feature_utility[target_variable]
    return global_utility, local_utility, per_feature_utility

In [15]:
perf_syn= compute_structural_fidelity_tabstruct(
    real_test,
    syn_df
)

perf_real= compute_structural_fidelity_tabstruct(
    real_test,
    real_train
)

Evaluating target: age
Evaluating target: workclass
Evaluating target: fnlwgt
Evaluating target: education
Evaluating target: educational-num
Evaluating target: marital-status
Evaluating target: occupation
Evaluating target: relationship
Evaluating target: race
Evaluating target: gender
Evaluating target: capital-gain
Evaluating target: capital-loss
Evaluating target: hours-per-week
Evaluating target: native-country
Evaluating target: income
Evaluating target: age
Evaluating target: workclass
Evaluating target: fnlwgt
Evaluating target: education
Evaluating target: educational-num
Evaluating target: marital-status
Evaluating target: occupation
Evaluating target: relationship
Evaluating target: race
Evaluating target: gender
Evaluating target: capital-gain
Evaluating target: capital-loss
Evaluating target: hours-per-week
Evaluating target: native-country
Evaluating target: income


In [16]:
print("Performance Scores Synthetic:", perf_syn)
print("Performance Scores Real:", perf_real)

Performance Scores Synthetic: {'age': np.float64(10.012923928790983), 'workclass': 0.7398562741846324, 'fnlwgt': np.float64(103106.4775140119), 'education': 0.8916528468767275, 'educational-num': np.float64(0.47274408445193333), 'marital-status': 0.8286346047540077, 'occupation': 0.3143173023770039, 'relationship': 0.7779988944168048, 'race': 0.8662244333886125, 'gender': 0.8442233278054173, 'capital-gain': np.float64(7100.0089895186475), 'capital-loss': np.float64(393.18796539773103), 'hours-per-week': np.float64(10.506282363595686), 'native-country': 0.760530679933665, 'income': 0.8558319513543394}
Performance Scores Real: {'age': np.float64(9.716009063279094), 'workclass': 0.7542288557213931, 'fnlwgt': np.float64(102203.38777671111), 'education': 1.0, 'educational-num': np.float64(0.11953752825811241), 'marital-status': 0.8512990602542841, 'occupation': 0.3589828634604754, 'relationship': 0.7992260917634052, 'race': 0.8824765063571034, 'gender': 0.8563847429519071, 'capital-gain': n

In [17]:
global_utility, local_utility, per_feature_utility=compute_utilities(perf_syn, perf_real, "income")
print(f"Global Utility: {global_utility}")
print(f"Local Utility: {local_utility}")
print(f"Per Feature Utility: {per_feature_utility}")

Global Utility: 0.9155002923281342
Local Utility: 0.980369807497467
Per Feature Utility: {'age': np.float64(0.9703468369855337), 'workclass': 0.9809440046907065, 'fnlwgt': np.float64(0.991241192997035), 'education': 0.8916528468767275, 'educational-num': np.float64(0.25285885575214745), 'marital-status': 0.9733766233766233, 'occupation': 0.8755774561133354, 'relationship': 0.9734403098630515, 'race': 0.9815835630167878, 'gender': 0.9857991221275496, 'capital-gain': np.float64(1.0011661060888581), 'capital-loss': np.float64(0.9998670420285382), 'hours-per-week': np.float64(0.9755348636534571), 'native-country': 0.8987457538541939, 'income': 0.980369807497467}
